# scMM 参数化处理与分析示例

本 notebook 将项目中的原始谱处理、数据清洗、注释、降维聚类、时间轨迹、代谢趋势和稳定性质控串成一个流程。通常只需修改下面的 **参数单元**，然后执行 `Run All`。

输入可为单个 `.mzML/.mzXML` 文件、包含多个原始文件的目录，或由 `CyESIData.save()` 生成且含 `.meta` 的结果目录。默认配置执行总离子流归一化和基础分析；耗时或需要额外数据的功能均通过开关启用。

In [ ]:
from pathlib import Path

# ==================== 只修改本单元 ====================
INPUT_PATH = Path("data/example.mzML")  # 原始文件、原始文件目录或已处理结果目录
INPUT_KIND = "auto"  # auto | raw_file | raw_dir | processed
OUTPUT_ROOT = Path("results")
FIGURE_DIR = OUTPUT_ROOT / "figures"
OVERWRITE = False
LOG_LEVEL = "INFO"

# 原始谱预处理；读取 processed 结果时这些参数不会使用
REF_MZ = 734.5929               # 用于识别细胞事件的参考离子 m/z，必须按实验修改
PPM_TOL = 10.0                  # 谱峰对齐容差
RESOLUTION_200 = 35_000.0       # Orbitrap 在 m/z 200 处的分辨率
RESAMPLE_POINTS_PER_FWHM = 5.0
MS_PEAK_SNR = 10.0              # 合并谱去噪阈值
CELL_SNR = 5.0                  # 参考离子细胞事件阈值
PEAK_SNR = 3.0                  # 单细胞内特征峰阈值
BASELINE_FILTER_SIZE = 50
MAX_ZERO_FRAC = 0.90
N_JOBS = -1

# 数据变换（按：去同位素 -> 异常值 -> 填补 -> 归一化 的顺序执行）
RUN_DEISOTOPE = False
DEISOTOPE_OPTIONS = dict(ppm_tol=1.0, max_isotope_order=3, r_square_threshold=0.95, merge_mode="keep_parent")
RUN_OUTLIER_REMOVAL = False
OUTLIER_OPTIONS = dict(contamination="auto", random_state=42)
IMPUTE_METHOD = None             # None | knn | mean | median | most_frequent
IMPUTE_OPTIONS = dict(n_neighbors=5)
NORMALIZATION = "total"       # None | total | max | quantile | pqn | zscore | log | minmax
NORMALIZATION_OPTIONS = dict(scale=1.0)

# 可选：LIPID MAPS 风格 SDF 精确质量注释
SDF_PATH = None                 # 例如 Path("data/structures.sdf")；不注释则为 None
ANNOTATION_PPM = 5.0
ION_MODE = "pos"              # pos | neg | both
MAX_ANNOTATIONS_PER_MZ = 5

# 可选：把原始目录内所有谱汇总为一个 total_sum_spec.mzML
EXPORT_SUMMED_SPECTRUM = False

# 下游分析
RUN_EMBEDDING = True
PCA_COMPONENTS = 50
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.7
RANDOM_STATE = 42
RUN_CLUSTERING = False          # 需要 leidenalg/python-igraph 或 networkx
CLUSTER_METHOD = "leiden"     # leiden | louvain
CLUSTER_RESOLUTION = 1.0

# 时间轨迹和趋势；原始数据通常自动具有归一化 time 列
PARAMETERIZATION_KEY = "time"
RUN_TRAJECTORY = True
RUN_METABOLIC_VELOCITY = False
RUN_METABOLITE_TRENDS = True
RUN_TREND_CLUSTERING = False
WINDOW_SIZE = 1000
STEP_SIZE = 300
PLOT_TOP_N = 30

# 可选稳定性质控和指定离子比值图
RUN_STABILITY_QC = False
TIME_BINS = 12
MONITOR_MZ = REF_MZ
RATIO_MZ = None                 # 例如 (768.5903, 774.6377)


## 1. 环境、输入识别与参数检查

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scMM.file.data import CyESIData

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL.upper()),
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

input_path = INPUT_PATH.expanduser().resolve()
output_root = OUTPUT_ROOT.expanduser().resolve()
figure_dir = FIGURE_DIR.expanduser().resolve()
if not input_path.exists():
    raise FileNotFoundError(f"输入不存在，请修改 INPUT_PATH：{input_path}")
if INPUT_KIND not in {"auto", "raw_file", "raw_dir", "processed"}:
    raise ValueError("INPUT_KIND 必须是 auto/raw_file/raw_dir/processed")

if INPUT_KIND == "auto":
    if input_path.is_file() and input_path.suffix.lower() in {".mzml", ".mzxml"}:
        input_kind = "raw_file"
    elif input_path.is_dir() and (input_path / ".meta").is_file():
        input_kind = "processed"
    elif input_path.is_dir():
        input_kind = "raw_dir"
    else:
        raise ValueError(f"无法自动识别输入类型：{input_path}")
else:
    input_kind = INPUT_KIND

if input_kind.startswith("raw") and (not np.isfinite(REF_MZ) or REF_MZ <= 0):
    raise ValueError("处理原始谱时 REF_MZ 必须为正的有限数")
output_root.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)
print(f"输入类型：{input_kind}\n输入：{input_path}\n输出：{output_root}")


## 2. 可选合并谱导出与数据载入

原始目录模式会先从所有文件的合并谱上检测公共 m/z 轴，再并行对齐各文件；单文件模式则直接完成相同的去噪、峰提取、帧对齐和细胞事件识别。

In [ ]:
if EXPORT_SUMMED_SPECTRUM:
    if input_kind != "raw_dir":
        raise ValueError("EXPORT_SUMMED_SPECTRUM 仅适用于原始文件目录")
    from joblib import Parallel, delayed
    from scMM.file.io import pack_specs, save_spectra, sum_spec, sum_spectrum_from_file

    raw_files = sorted(p for p in input_path.iterdir() if p.is_file() and p.suffix.lower() in {".mzml", ".mzxml"})
    if not raw_files:
        raise FileNotFoundError(f"目录中没有 mzML/mzXML：{input_path}")
    per_file_specs = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(sum_spectrum_from_file)(p, resolution_200=RESOLUTION_200, points_per_fwhm=RESAMPLE_POINTS_PER_FWHM)
        for p in raw_files
    )
    total_spec = sum_spec(pack_specs(per_file_specs), resolution_200=RESOLUTION_200, points_per_fwhm=RESAMPLE_POINTS_PER_FWHM)
    summed_path = save_spectra(total_spec, output_root / "total_sum_spec.mzML")
    print(f"合并谱已保存：{summed_path}")

load_options = dict(
    ref_mz=REF_MZ,
    ppm_tol=PPM_TOL,
    resolution=RESOLUTION_200,
    resample_points_per_fwhm=RESAMPLE_POINTS_PER_FWHM,
    ms_peak_snr_threshold=MS_PEAK_SNR,
    cell_snr=CELL_SNR,
    peak_snr=PEAK_SNR,
    baseline_filter_size=BASELINE_FILTER_SIZE,
    max_zero_frac=MAX_ZERO_FRAC,
)
if input_kind == "processed":
    data = CyESIData.load_from_processed(input_path)
elif input_kind == "raw_file":
    data = CyESIData.load_from_file(input_path, **load_options)
else:
    data = CyESIData.load_from_filelist(input_path, n_jobs=N_JOBS, **load_options)

if len(data) == 0 or data.data.shape[1] == 0:
    raise ValueError("没有识别到细胞或特征；请检查 REF_MZ、SNR、分辨率和 ppm 参数")
print(f"已载入 {data.get_name()}：{len(data):,} 个细胞 × {data.data.shape[1]:,} 个特征")


## 3. 数据变换、可选注释与保存

In [ ]:
if RUN_DEISOTOPE:
    before = data.data.shape[1]
    data.deisotope(**DEISOTOPE_OPTIONS)
    print(f"去同位素：{before:,} -> {data.data.shape[1]:,} 个特征")
if RUN_OUTLIER_REMOVAL:
    before = len(data)
    data.remove_outlier(**OUTLIER_OPTIONS)
    print(f"异常值过滤：{before:,} -> {len(data):,} 个细胞")
if IMPUTE_METHOD is not None:
    options = dict(IMPUTE_OPTIONS)
    if IMPUTE_METHOD != "knn":
        options.pop("n_neighbors", None)
    data.impute(method=IMPUTE_METHOD, missing_values=0, **options)
if NORMALIZATION is not None:
    data.normalize(method=NORMALIZATION, **NORMALIZATION_OPTIONS)

annotation_hits = None
if SDF_PATH is not None:
    sdf_path = Path(SDF_PATH).expanduser().resolve()
    if not sdf_path.is_file():
        raise FileNotFoundError(f"SDF_PATH 不存在：{sdf_path}")
    annotation_hits = data.get_annotation(
        sdf_path, ppm_tol=ANNOTATION_PPM, search_mode=ION_MODE, max_results_per_mz=MAX_ANNOTATIONS_PER_MZ
    )
    annotation_hits.to_csv(output_root / "annotation_candidates.csv", index=False)
    if not annotation_hits.empty:
        best = annotation_hits.sort_values("abs_ppm_error").drop_duplicates("query_mz").set_index("query_mz")
        feature_mz = data.feature_meta["mz"].astype(float)
        for source, target in {"COMMON_NAME": "annotation_name", "FORMULA": "annotation_formula", "adduct": "annotation_adduct", "ppm_error": "annotation_ppm_error"}.items():
            data.feature_meta[target] = feature_mz.map(best[source])
    print(f"注释候选：{len(annotation_hits):,} 条")

result_path = data.save(output_root, overwrite=OVERWRITE)
summary = pd.Series({
    "dataset": data.get_name(), "cells": len(data), "features": data.data.shape[1],
    "zero_fraction": float((data.data == 0).to_numpy().mean()),
    "normalization": NORMALIZATION or "none", "result_path": str(result_path),
})
display(summary.to_frame("value"))


## 4. AnnData、PCA/UMAP 与可选聚类

`obs` 保存每个细胞的保留时间、归一化时间和来源标签；`var` 保存 m/z、去同位素信息及可选注释；强度矩阵位于 `X`。

In [ ]:
from scMM.plot.engine import PlotEngine

adata = data.to_anndata()
engine = PlotEngine.from_adata(adata, fig_path_dir=figure_dir)
if RUN_EMBEDDING:
    if engine.adata.n_obs < 3:
        raise ValueError("UMAP 至少需要 3 个细胞")
    engine.pca(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    neighbors = min(max(2, UMAP_NEIGHBORS), engine.adata.n_obs - 1)
    embedding = engine.umap(use_pca=True, n_neighbors=neighbors, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)
    color_key = PARAMETERIZATION_KEY if PARAMETERIZATION_KEY in engine.adata.obs else None
    fig, ax = plt.subplots(figsize=(6, 6))
    colors = engine.adata.obs[color_key].astype(float) if color_key else "#4C78A8"
    scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=colors, s=3, cmap="viridis", linewidths=0)
    if color_key:
        fig.colorbar(scatter, ax=ax, label=color_key)
    ax.set(xlabel="UMAP 1", ylabel="UMAP 2")
    fig.savefig(figure_dir / "umap.svg", bbox_inches="tight")
    plt.show()
    if RUN_CLUSTERING:
        engine.cluster_cells(method=CLUSTER_METHOD, n_neighbors=neighbors, resolution=CLUSTER_RESOLUTION, random_state=RANDOM_STATE)
elif RUN_CLUSTERING or RUN_TRAJECTORY:
    raise ValueError("聚类和轨迹分析依赖 UMAP，请启用 RUN_EMBEDDING")


## 5. 时间轨迹、代谢速度与特征趋势

In [ ]:
needs_time = RUN_TRAJECTORY or RUN_METABOLIC_VELOCITY or RUN_METABOLITE_TRENDS
if needs_time and PARAMETERIZATION_KEY not in engine.adata.obs:
    raise KeyError(f"obs 中没有时间参数列 {PARAMETERIZATION_KEY!r}")
if needs_time and not np.isfinite(engine.adata.obs[PARAMETERIZATION_KEY].astype(float)).all():
    raise ValueError(f"{PARAMETERIZATION_KEY} 含非有限值")

window = min(WINDOW_SIZE, engine.adata.n_obs)
step = min(max(1, STEP_SIZE), window)
if RUN_TRAJECTORY:
    engine.compute_trajectory(
        window_size=window, step_size=step, parameterization_key=PARAMETERIZATION_KEY,
        branch_prob_key=None, min_cells_per_window=min(5, window), plotting=True, cmap="Spectral_r",
    )
if RUN_METABOLIC_VELOCITY:
    engine.metabolic_velocity(window_size=max(2, window), step_size=step, parameterization_key=PARAMETERIZATION_KEY)
if RUN_METABOLITE_TRENDS:
    feature_name_key = "annotation_name" if "annotation_name" in engine.adata.var else "mz"
    engine.plot_metabolite_trends(
        parameterization_key=PARAMETERIZATION_KEY, window_size=window, step_size=step,
        kernel_stat="median", feature_name_key=feature_name_key, plot_top_n=min(PLOT_TOP_N, engine.adata.n_vars),
        cmap="viridis", xlabel=PARAMETERIZATION_KEY, ylabel=feature_name_key,
    )
    if RUN_TREND_CLUSTERING:
        engine.plot_trend_clusters(metric="correlation", cluster_method="leiden", top_k=min(PLOT_TOP_N, engine.adata.n_vars))


## 6. 可选稳定性质控与离子比值图

In [ ]:
if RUN_STABILITY_QC:
    import seaborn as sns

    if PARAMETERIZATION_KEY not in data.peak_meta:
        raise KeyError(f"peak_meta 中没有 {PARAMETERIZATION_KEY!r}")
    times = data.peak_meta[PARAMETERIZATION_KEY].astype(float).to_numpy()
    edges = np.linspace(np.nanmin(times), np.nanmax(times), TIME_BINS + 1)
    labels = np.clip(np.digitize(times, edges[1:-1]) + 1, 1, TIME_BINS)
    mz_values = data.data.columns.astype(float).to_numpy()
    monitor_col = data.data.columns[np.argmin(np.abs(mz_values - MONITOR_MZ))]
    qc = pd.DataFrame({"time_bin": labels, "intensity": data.data[monitor_col].to_numpy()})
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=qc, x="time_bin", y="intensity", ax=ax, color="#E28962", fliersize=2)
    ax.set_ylabel(f"Intensity near m/z {float(monitor_col):.4f}")
    fig.savefig(figure_dir / "stability_monitor_boxplot.svg", bbox_inches="tight")
    plt.close(fig)

    cell_counts = pd.Series(labels).value_counts().reindex(range(1, TIME_BINS + 1), fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(cell_counts.index, cell_counts.values, color="#5A9DA4")
    ax.set(xlabel="Time bin", ylabel="Cell count")
    fig.savefig(figure_dir / "stability_cell_counts.svg", bbox_inches="tight")
    plt.close(fig)

    medians = np.vstack([data.data.iloc[labels == i].median(axis=0).to_numpy() for i in range(1, TIME_BINS + 1) if np.any(labels == i)])
    if medians.shape[0] >= 2:
        corr = np.corrcoef(medians)
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(corr, cmap="Oranges", vmin=-1, vmax=1, square=True, ax=ax)
        ax.set_title("Median-profile Pearson correlation")
        fig.savefig(figure_dir / "stability_median_correlation.svg", bbox_inches="tight")
        plt.close(fig)

if RATIO_MZ is not None:
    numerator, denominator = map(float, RATIO_MZ)
    ratio = np.divide(data[numerator], data[denominator], out=np.full(len(data), np.nan), where=data[denominator] != 0)
    if "X_umap" not in engine.adata.obsm:
        raise KeyError("离子比值图需要先运行 UMAP")
    xy = engine.adata.obsm["X_umap"]
    fig, ax = plt.subplots(figsize=(6, 6))
    scatter = ax.scatter(xy[:, 0], xy[:, 1], c=ratio, s=3, cmap="viridis", linewidths=0)
    fig.colorbar(scatter, ax=ax, label=f"{numerator:.4f} / {denominator:.4f}")
    fig.savefig(figure_dir / "feature_ratio_umap.svg", bbox_inches="tight")
    plt.show()


## 输出说明

处理结果位于 `OUTPUT_ROOT/<数据集名>/`：`.meta` 记录数据集与处理元数据，`data` 是细胞×m/z 强度矩阵，`peak_meta` 是细胞元数据，`feature_meta` 是特征及注释元数据；每张表同时保存为高保真/快速加载的 pickle 和便于交换的 CSV。所有分析图写入 `FIGURE_DIR`，SDF 的完整候选命中另存为 `annotation_candidates.csv`。